# Universal 2D to 3D Pipeline

Run the setup cell first, then use the pipeline cell to process images.

In [ ]:
# @title 🛠️ Step 1: Environment Setup
# Run this cell ONCE to install all required dependencies.

import os
import subprocess

print("Mounting Google Drive...")
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=True)
except Exception as e:
    print(f"Drive mount failed: {e}")

print("Installing dependencies and cloning repositories...")
if not os.path.exists('/content/Hunyuan3D-2'):
    !git clone --recurse-submodules https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/Hunyuan3D-2

%cd /content/Hunyuan3D-2
!pip install -q --upgrade pip
!pip install -q tokenizers==0.21.0 transformers==4.48.2 --prefer-binary
!pip install -q "jedi>=0.16" "numpy<2.5" "cuda-bindings~=13.0.3" "numba<=0.65.1,>=0.58" "rich<14" "fsspec[http]<=2026.2.0,>=2023.1.0" "huggingface_hub<2.0" trimesh rtree pymeshlab accelerate diffusers datasets torchvision 
!pip install -q -U "pillow==10.4.0"
!pip install -q -r requirements.txt
!pip install -q -e .

# Install Rust toolchain for SolidCraft
if not os.path.exists(os.path.expanduser('~/.cargo/bin/cargo')):
    print("Installing Rust toolchain...")
    !curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y

os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"

# Clone and build SolidCraft
solidcraft_dir = "/content/solidcraft"
if not os.path.exists(solidcraft_dir):
    print("Cloning SolidCraft repository...")
    !git clone https://github.com/tangency-labs/solidcraft.git {solidcraft_dir}

print("Building SolidCraft release binary...")
subprocess.run(["cargo", "build", "--release", "-p", "solidcraft-cli"], cwd=solidcraft_dir, check=True)

print("\n✅ Environment setup complete! You can now run the pipeline.")

print("\n🔄 Restarting runtime to load new packages...")
import os
os.kill(os.getpid(), 9)


In [ ]:
# @title 🚀 Step 2: 2D Image to 3D Model Pipeline
# Modify the settings below and run the cell to generate a 3D model.

import os
import sys
import subprocess
import importlib
import PIL.Image
import torch
from google.colab import files

# --- Configurable Parameters ---
project_folder = "3D Print Projects" # @param {type:"string"}
input_image_name = "orca_3quarter_view.png" # @param {type:"string"}
output_model_name = "orca_model" # @param {type:"string"}
export_only_raw_obj = False # @param {type:"boolean"}

drive_base = "/content/drive/MyDrive"
project_dir = os.path.join(drive_base, project_folder)
os.makedirs(project_dir, exist_ok=True)

input_image_path = os.path.join(project_dir, input_image_name)
raw_obj_path = os.path.join(project_dir, f"{output_model_name}_raw.obj")
clean_obj_path = os.path.join(project_dir, f"{output_model_name}_clean.obj")
final_output_path = os.path.join(project_dir, f"{output_model_name}_print_ready.3mf")

if not os.path.exists(input_image_path):
    print(f"❌ ERROR: Input image not found at '{input_image_path}'. Please upload it to this folder in Google Drive.")
    sys.exit(1)

print("=== STEP 1: Running Hunyuan3D Synthesis ===")
# Ensure pipeline imports
if not hasattr(PIL.Image, "ANTIALIAS"):
    PIL.Image.ANTIALIAS = PIL.Image.LANCZOS
repo_path = "/content/Hunyuan3D-2"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

try:
    from hy3dgen.shapegen import Hunyuan3DDiTFlowMatchingPipeline
    if 'hunyuan_pipeline' not in globals():
        print("Loading Hunyuan3D pipeline (this takes a moment)...")
        global hunyuan_pipeline
        hunyuan_pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
            "tencent/Hunyuan3D-2", torch_dtype=torch.float16, device="cuda"
        )
    else:
        print("Using cached Hunyuan3D pipeline.")
    pipeline = hunyuan_pipeline
    print(f"Synthesizing 3D mesh for '{input_image_name}'...")
    mesh = pipeline(image=input_image_path)[0]
    mesh.export(raw_obj_path)
    print(f"✅ Raw mesh saved to {raw_obj_path}")
except Exception as e:
    print(f"\n❌ ERROR during synthesis: {e}")
    sys.exit(1)

if export_only_raw_obj:
    print("\n✅ Process complete! Exporting raw OBJ only as requested.")
    print(f"File location: {raw_obj_path}")
else:
    print("\n=== STEP 2: Pre-processing & Repairing Mesh ===")
    import trimesh
    mesh = trimesh.load(raw_obj_path, force='mesh')
    trimesh.repair.fill_holes(mesh)
    trimesh.repair.fix_normals(mesh)
    trimesh.repair.fix_inversion(mesh)

    if len(mesh.faces) > 50000:
        print(f"Decimating mesh faces from {len(mesh.faces):,} to 50,000...")
        mesh = mesh.simplify_quadric_decimation(face_count=50000)

    mesh.export(clean_obj_path)
    print(f"Cleaned mesh saved (Watertight: {mesh.is_watertight})")

    print("\n=== STEP 3: Processing through SolidCraft ===")
    os.environ["PATH"] += f":{os.path.expanduser('~')}/.cargo/bin"
    solidcraft_cli = "/content/solidcraft/target/release/solidcraft-cli"

    cmd = [
        solidcraft_cli,
        "run",
        "--input", clean_obj_path,
        "--out", final_output_path
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0 and os.path.exists(final_output_path):
        print(f"\n✅ Success! Print-ready file generated at: {final_output_path}")
        print(f"File size: {os.path.getsize(final_output_path) / (1024 * 1024):.2f} MB")
    else:
        print("SolidCraft CLI execution notice/output:", result.stdout)
        if result.stderr:
            print("Stderr details:", result.stderr)
